# Backbone Experiment: dino_vits8

## Setup

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))  # repo root
from config import Config
from data.cifake import get_cifake_loaders
from experiments.train import train
from experiments.evaluate import full_evaluation
from experiments.baseline_spatial_only import run_spatial_only_baseline
from tqdm.notebook import tqdm
import pandas as pd
import torch

print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
if torch.cuda.is_available():
    print(f"GPU:    {torch.cuda.get_device_name(0)}")
    print(f"VRAM:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


Device: cuda
GPU:    NVIDIA RTX PRO 4000 Blackwell
VRAM:   25.2 GB


## Shared Config

In [2]:
# Only line to change
BACKBONE = "dino_vits8"

def make_cfg(fusion_mode, frozen=False):
    cfg = Config()
    cfg.data.cifake_root    = "../data/raw/cifake"
    cfg.data.num_workers    = 4 # change to 0 if on Windows
    cfg.data.batch_size     = 64
    cfg.data.jpeg_aug       = True
    cfg.backbone.name       = BACKBONE
    cfg.backbone.pretrained = True
    cfg.backbone.frozen     = frozen
    cfg.backbone.img_size   = cfg.data.image_size  
    cfg.fusion.mode         = fusion_mode
    cfg.train.epochs        = 50
    cfg.train.backbone_lr   = 1e-4  
    cfg.train.lr            = 1e-4
    cfg.experiment_name     = f"{BACKBONE}_{fusion_mode}{'_frozen' if frozen else ''}"
    cfg.notes               = f"CIFAKE · {BACKBONE} · {fusion_mode}{'_frozen' if frozen else ''}"
    return cfg

# Load data
cfg_data = make_cfg("joint_only") # backbone settings only, fusion mode ignored
train_loader, val_loader, test_loader = get_cifake_loaders(cfg_data)
print(f"Train: {len(train_loader.dataset):,}  Val: {len(val_loader.dataset):,}  Test: {len(test_loader.dataset):,}")

Train: 85,000  Val: 15,000  Test: 20,000
Train: 85,000  Val: 15,000  Test: 20,000


## Experiment 0: Spatial-Only Baseline
Trains only the spatial backbone as a standalone classifier with no frequency branch.
This gives the honest floor. How well the backbone alone can classify real vs fake.
All delta values in later experiments are relative to this number.


In [3]:
cfg0 = make_cfg("joint_only")  # backbone settings only, fusion mode ignored
cfg0.experiment_name = f"{BACKBONE}_spatial_only"
cfg0.notes           = f"spatial-only baseline, no freq branch, {BACKBONE}"
spatial_acc = run_spatial_only_baseline(cfg0, train_loader, val_loader, test_loader)
print(f"\nSpatial-only floor: {spatial_acc:.1%}")
print("All subsequent delta values are relative to this number.")

Device: cuda


Training spatial-only baseline (dino_vits8) for 50 epochs...
Train: 85,000  Val: 15,000


Epoch   1/50 | train_loss=0.2104 | val_acc=94.5%


Epoch   2/50 | train_loss=0.1518 | val_acc=93.7%


Epoch   3/50 | train_loss=0.1349 | val_acc=96.1%


...

Epoch  46/50 | train_loss=0.0021 | val_acc=98.3%


Epoch  47/50 | train_loss=0.0020 | val_acc=98.4%


Epoch  48/50 | train_loss=0.0023 | val_acc=98.4%


Epoch  49/50 | train_loss=0.0022 | val_acc=98.3%


Epoch  50/50 | train_loss=0.0011 | val_acc=98.4%

Spatial-only results (dino_vits8):
  Accuracy: 98.2%
  AUC-ROC:  0.997
  F1:       0.982
Results saved → ./results/results.csv  (dino_vits8_spatial_only, acc=0.9818)
Results saved to ./results/results.csv

Spatial-only floor: 98.2%
All subsequent delta values are relative to this number.


## Experiment 1: Joint-Only Baseline
Both branches concatenated into a single shared classifier. No weighting between branches.
This is the floor — all other experiments are compared against this delta value.


In [4]:
cfg1 = make_cfg("joint_only")
print(f"Running: {cfg1.experiment_name}")
train(cfg1, train_loader, val_loader, test_loader)

Running: dino_vits8_joint_only
Device: cuda
GPU: NVIDIA RTX PRO 4000 Blackwell

Experiment: dino_vits8_joint_only
Backbone: dino_vits8 | Fusion: joint_only | Frozen: False
Epochs: 50 | LR: 0.0001 | Batch: 64



Epoch   1/50 | train_loss=0.4757 | val_acc=94.1% | val_auc=0.991 | val_f1=0.943
  grad norms — freq=0.2099 | spatial=0.9673
  -> Saved best val_acc=94.1%


Epoch   2/50 | train_loss=0.3609 | val_acc=92.7% | val_auc=0.992 | val_f1=0.932


Epoch   3/50 | train_loss=0.3288 | val_acc=95.0% | val_auc=0.993 | val_f1=0.952
  -> Saved best val_acc=95.0%


Epoch   4/50 | train_loss=0.3028 | val_acc=96.3% | val_auc=0.995 | val_f1=0.963
  -> Saved best val_acc=96.3%


...

Epoch  47/50 | train_loss=0.0876 | val_acc=98.2% | val_auc=0.998 | val_f1=0.982


Epoch  48/50 | train_loss=0.0865 | val_acc=98.2% | val_auc=0.998 | val_f1=0.982


Epoch  49/50 | train_loss=0.0860 | val_acc=98.3% | val_auc=0.998 | val_f1=0.983


Epoch  50/50 | train_loss=0.0856 | val_acc=98.3% | val_auc=0.998 | val_f1=0.983

Training complete. Best val accuracy: 98.3%
Results will be logged to CSV after full_evaluation().


0.9832666666666666

In [5]:
results1 = full_evaluation(
    cfg1,
    checkpoint_path=f"./checkpoints/best_{cfg1.experiment_name}.pt",
    test_loader=test_loader,
)

Loaded checkpoint: ./checkpoints/best_dino_vits8_joint_only.pt

EVALUATION — dino_vits8_joint_only
Backbone: dino_vits8 | Fusion: joint_only | Frozen: False
  Joint accuracy:     98.3%
  AUC-ROC:            0.998
  F1:                 0.983
  Spatial-only:       98.1%
  Freq-only:          92.6%
  Delta (Δ):          +0.2%  (freq branch contribution)

  No warning signs triggered. ✓
Results saved → ./results/results.csv  (dino_vits8_joint_only, acc=0.9829)


## Experiment 2: Scalar Fusion
Two learned scalars (a, b) weight each branch. Softmax ensures a+b=1 always.
Watch the scalar values logged each epoch — b is the frequency weight.
If b drops below 0.1 early in training the frequency branch is being ignored.


In [6]:
cfg2 = make_cfg("scalar")
print(f"Running: {cfg2.experiment_name}")
train(cfg2, train_loader, val_loader, test_loader)

Running: dino_vits8_scalar
Device: cuda
GPU: NVIDIA RTX PRO 4000 Blackwell

Experiment: dino_vits8_scalar
Backbone: dino_vits8 | Fusion: scalar | Frozen: False
Epochs: 50 | LR: 0.0001 | Batch: 64



Epoch   1/50 | train_loss=0.4582 | val_acc=95.4% | val_auc=0.991 | val_f1=0.953
  scalars — spatial=0.492 freq=0.508
  grad norms — freq=0.3396 | spatial=0.9346
  -> Saved best val_acc=95.4%


Epoch   2/50 | train_loss=0.3540 | val_acc=96.2% | val_auc=0.993 | val_f1=0.962
  scalars — spatial=0.489 freq=0.511
  -> Saved best val_acc=96.2%


Epoch   3/50 | train_loss=0.3196 | val_acc=95.6% | val_auc=0.994 | val_f1=0.958
  scalars — spatial=0.488 freq=0.512


Epoch   4/50 | train_loss=0.2939 | val_acc=94.4% | val_auc=0.994 | val_f1=0.946
  scalars — spatial=0.486 freq=0.514


...

Epoch  47/50 | train_loss=0.0851 | val_acc=98.4% | val_auc=0.999 | val_f1=0.984
  scalars — spatial=0.473 freq=0.527
  -> Saved best val_acc=98.4%


Epoch  48/50 | train_loss=0.0842 | val_acc=98.4% | val_auc=0.998 | val_f1=0.984
  scalars — spatial=0.473 freq=0.527


Epoch  49/50 | train_loss=0.0850 | val_acc=98.4% | val_auc=0.998 | val_f1=0.984
  scalars — spatial=0.473 freq=0.527


Epoch  50/50 | train_loss=0.0833 | val_acc=98.4% | val_auc=0.998 | val_f1=0.984
  scalars — spatial=0.473 freq=0.527

Training complete. Best val accuracy: 98.4%
Results will be logged to CSV after full_evaluation().


0.9840666666666666

In [7]:
results2 = full_evaluation(
    cfg2,
    checkpoint_path=f"./checkpoints/best_{cfg2.experiment_name}.pt",
    test_loader=test_loader,
)

Loaded checkpoint: ./checkpoints/best_dino_vits8_scalar.pt

EVALUATION — dino_vits8_scalar
Backbone: dino_vits8 | Fusion: scalar | Frozen: False
  Joint accuracy:     98.3%
  AUC-ROC:            0.998
  F1:                 0.984
  Spatial-only:       98.2%
  Freq-only:          92.1%
  Delta (Δ):          +0.1%  (freq branch contribution)

  No warning signs triggered. ✓
Results saved → ./results/results.csv  (dino_vits8_scalar, acc=0.9835)


## Experiment 3: Gating Fusion 
A small MLP outputs a per-image gate value in [0,1] controlling how much to trust the frequency branch.
Key metric beyond accuracy: **gate entropy must be > 0.3 nats**.
Below that the gate is outputting near-constant values and is not genuinely adapting per sample.


In [8]:
cfg3 = make_cfg("gating")
print(f"Running: {cfg3.experiment_name}")
train(cfg3, train_loader, val_loader, test_loader)

Running: dino_vits8_gating
Device: cuda
GPU: NVIDIA RTX PRO 4000 Blackwell

Experiment: dino_vits8_gating
Backbone: dino_vits8 | Fusion: gating | Frozen: False
Epochs: 50 | LR: 0.0001 | Batch: 64



Epoch   1/50 | train_loss=0.5408 | val_acc=92.2% | val_auc=0.991 | val_f1=0.927
  gate — entropy=1.036 nats | mean=0.682 | var=0.0010
  grad norms — freq=0.3037 | spatial=0.9468
  -> Saved best val_acc=92.2%


Epoch   2/50 | train_loss=0.4354 | val_acc=95.3% | val_auc=0.991 | val_f1=0.953
  gate — entropy=1.405 nats | mean=0.589 | var=0.0023
  -> Saved best val_acc=95.3%


Epoch   3/50 | train_loss=0.3936 | val_acc=96.2% | val_auc=0.994 | val_f1=0.962
  gate — entropy=1.231 nats | mean=0.604 | var=0.0015
  -> Saved best val_acc=96.2%


Epoch   4/50 | train_loss=0.3850 | val_acc=96.5% | val_auc=0.994 | val_f1=0.965
  gate — entropy=1.150 nats | mean=0.647 | var=0.0013
  -> Saved best val_acc=96.5%


...

Epoch  47/50 | train_loss=0.1253 | val_acc=98.4% | val_auc=0.998 | val_f1=0.984
  gate — entropy=1.705 nats | mean=0.533 | var=0.0043


Epoch  48/50 | train_loss=0.1223 | val_acc=98.3% | val_auc=0.998 | val_f1=0.984
  gate — entropy=1.749 nats | mean=0.520 | var=0.0047


Epoch  49/50 | train_loss=0.1192 | val_acc=98.3% | val_auc=0.998 | val_f1=0.983
  gate — entropy=1.788 nats | mean=0.515 | var=0.0051


Epoch  50/50 | train_loss=0.1161 | val_acc=98.4% | val_auc=0.998 | val_f1=0.984
  gate — entropy=1.813 nats | mean=0.507 | var=0.0054

Training complete. Best val accuracy: 98.4%
Results will be logged to CSV after full_evaluation().


0.9841333333333333

In [9]:
results3 = full_evaluation(
    cfg3,
    checkpoint_path=f"./checkpoints/best_{cfg3.experiment_name}.pt",
    test_loader=test_loader,
)

Loaded checkpoint: ./checkpoints/best_dino_vits8_gating.pt

EVALUATION — dino_vits8_gating
Backbone: dino_vits8 | Fusion: gating | Frozen: False
  Joint accuracy:     98.3%
  AUC-ROC:            0.998
  F1:                 0.983
  Spatial-only:       98.2%
  Freq-only:          91.8%
  Delta (Δ):          +0.1%  (freq branch contribution)

  Gate distribution:
    entropy:  1.790 nats  (OK)
    mean:     0.535
    variance: 0.0052

  No warning signs triggered. ✓
Results saved → ./results/results.csv  (dino_vits8_gating, acc=0.9829)


## Experiment 4: Gating Fusion, Frozen Backbone
Same as Experiment 3 but backbone weights are locked.
Only the projection head, frequency branch, fusion, and joint head train.

If the frequency branch helps more here than in Experiment 3, it means the backbone
was learning to capture spectral information during fine-tuning in Experiment 3 —
essentially teaching itself what the frequency branch provides.


In [10]:
cfg4 = make_cfg("gating", frozen=True)
print(f"Running: {cfg4.experiment_name}")
train(cfg4, train_loader, val_loader, test_loader)

Running: dino_vits8_gating_frozen
Device: cuda
GPU: NVIDIA RTX PRO 4000 Blackwell

Experiment: dino_vits8_gating_frozen
Backbone: dino_vits8 | Fusion: gating | Frozen: True
Epochs: 50 | LR: 0.0001 | Batch: 64



Epoch   1/50 | train_loss=0.5371 | val_acc=89.5% | val_auc=0.979 | val_f1=0.903
  gate — entropy=1.988 nats | mean=0.604 | var=0.0077
  grad norms — freq=nan | spatial=nan
  -> Saved best val_acc=89.5%


Epoch   2/50 | train_loss=0.4404 | val_acc=92.0% | val_auc=0.984 | val_f1=0.924
  gate — entropy=2.278 nats | mean=0.662 | var=0.0145
  -> Saved best val_acc=92.0%


Epoch   3/50 | train_loss=0.3979 | val_acc=94.2% | val_auc=0.987 | val_f1=0.943
  gate — entropy=2.461 nats | mean=0.681 | var=0.0226
  -> Saved best val_acc=94.2%


Epoch   4/50 | train_loss=0.3782 | val_acc=94.5% | val_auc=0.988 | val_f1=0.946
  gate — entropy=2.508 nats | mean=0.696 | var=0.0264
  -> Saved best val_acc=94.5%


...

Epoch  47/50 | train_loss=0.2337 | val_acc=96.8% | val_auc=0.995 | val_f1=0.968
  gate — entropy=2.777 nats | mean=0.634 | var=0.0749


Epoch  48/50 | train_loss=0.2297 | val_acc=96.8% | val_auc=0.995 | val_f1=0.968
  gate — entropy=2.778 nats | mean=0.633 | var=0.0754


Epoch  49/50 | train_loss=0.2313 | val_acc=96.8% | val_auc=0.995 | val_f1=0.968
  gate — entropy=2.787 nats | mean=0.627 | var=0.0762


Epoch  50/50 | train_loss=0.2305 | val_acc=96.9% | val_auc=0.995 | val_f1=0.969
  gate — entropy=2.791 nats | mean=0.625 | var=0.0771

Training complete. Best val accuracy: 96.9%
Results will be logged to CSV after full_evaluation().


0.9692

In [11]:
results4 = full_evaluation(
    cfg4,
    checkpoint_path=f"./checkpoints/best_{cfg4.experiment_name}.pt",
    test_loader=test_loader,
)

Loaded checkpoint: ./checkpoints/best_dino_vits8_gating_frozen.pt

EVALUATION — dino_vits8_gating_frozen
Backbone: dino_vits8 | Fusion: gating | Frozen: True
  Joint accuracy:     96.9%
  AUC-ROC:            0.996
  F1:                 0.969
  Spatial-only:       86.6%
  Freq-only:          92.0%
  Delta (Δ):          +10.3%  (freq branch contribution)

  Gate distribution:
    entropy:  2.779 nats  (OK)
    mean:     0.631
    variance: 0.0764

  No warning signs triggered. ✓
Results saved → ./results/results.csv  (dino_vits8_gating_frozen, acc=0.969)


## Summary: dino_vits8
Compare all four runs. Key columns: accuracy, delta (freq branch contribution), gate_entropy.


In [12]:
df = pd.read_csv("./results/results.csv")
mask = df["backbone"] == BACKBONE
cols = ["experiment_name", "fusion", "frozen", "accuracy", "auc_roc", "f1", "gate_entropy"]
print(df[mask][cols].to_string(index=False))

         experiment_name     fusion  frozen  accuracy  auc_roc     f1  gate_entropy
 dino_vits8_spatial_only joint_only   False    0.9818   0.9972 0.9819        0.0000
   dino_vits8_joint_only joint_only   False    0.9829   0.9981 0.9829        0.0000
       dino_vits8_scalar     scalar   False    0.9835   0.9984 0.9835        0.0000
       dino_vits8_gating     gating   False    0.9829   0.9983 0.9829        1.7902
dino_vits8_gating_frozen     gating    True    0.9690   0.9958 0.9693        2.7787


**What to look for:**
- **Delta** (printed by full_evaluation) - how much the frequency branch added over spatial alone
- **Gate entropy** - must be > 0.3 nats for gating experiments to be valid
- **Frozen vs fine-tuned** - if frozen delta > fine-tuned delta, the backbone was absorbing spectral information during fine-tuning
